In [1]:
from pathlib import Path
import pandas as pd

Path.cwd()

WindowsPath('c:/Users/HP/OneDrive/Master/Desarrollo de soluciones de IA/dsia-26-27-lamamie_de_clairac-tatiana/Ejercicios')

In [2]:
ruta_ventas = Path("../Datos/ventas.csv")
ruta_ventas

WindowsPath('../Datos/ventas.csv')

In [3]:
ventas = pd.read_csv(ruta_ventas)
ventas

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-05,Norte,Sensor A,12.0,19.5,C001
1,2026-01-05,Sur,Sensor B,4.0,32.0,C002
2,2026-01-06,Norte,Sensor A,7.0,19.5,C003
3,2026-01-07,Este,Gateway,2.0,120.0,C001
4,2026-01-08,Sur,Sensor B,NaN,32.0,C004
5,2026-01-09,Oeste,Sensor A,15.0,19.5,C005
6,2026-01-10,Norte,Gateway,1.0,120.0,C002
7,2026-01-10,Este,Sensor B,9.0,32.0,C006
8,2026-01-11,Sur,Sensor A,3.0,-5.0,C007
9,2026-01-12,Norte,Sensor A,8.0,19.5,C001


In [4]:
print(ventas.shape)

print(ventas.dtypes)

print(ventas.isna().sum())

(10, 6)
fecha                  str
region                 str
producto               str
unidades           float64
precio_unitario    float64
cliente_id             str
dtype: object
fecha              0
region             0
producto           0
unidades           1
precio_unitario    0
cliente_id         0
dtype: int64


In [5]:
unidades_num = pd.to_numeric(ventas["unidades"], errors="coerce") # Si pandas encuentra algo que no puede convertir a número, por ejemplo, "abc", en vez de dar error lo convierte en NaN. Así podemos detectarlo con .isna()
precio_num = pd.to_numeric(ventas["precio_unitario"], errors="coerce")

filas_invalidas = ventas[
    unidades_num.isna() | (unidades_num <= 0) |
    precio_num.isna() | (precio_num <= 0)
]

filas_invalidas

,fecha,region,producto,unidades,precio_unitario,cliente_id
4,2026-01-08,Sur,Sensor B,NaN,32.0,C004
8,2026-01-11,Sur,Sensor A,3.0,-5.0,C007


In [6]:
# Implementar la función validar_ventas que devuelva dos dataframes: validos y errores

def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = frame.copy()

    df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
    df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")

    mascara_validos = (
        df["unidades"].notna()
        & (df["unidades"] > 0)
        & df["precio_unitario"].notna()
        & (df["precio_unitario"] > 0)
    )

    validos = df[mascara_validos].copy()
    errores = df[~mascara_validos].copy()

    validos["importe"] = validos["unidades"] * validos["precio_unitario"]

    return validos, errores

In [7]:
validos, errores = validar_ventas(ventas)

print("Filas válidas:", len(validos))
print("Filas inválidas:", len(errores))

Filas válidas: 8
Filas inválidas: 2


In [8]:
# Agregaciones. Calcular el importe total por región, ordenado de mayor a menor
importe_por_region = (
    validos
    .groupby("region")["importe"]
    .sum()
    .sort_values(ascending=False)
)

importe_por_region

region
Norte    646.5
Este     528.0
Oeste    292.5
Sur      128.0
Name: importe, dtype: float64

In [10]:
# Sacar top 3 productos por importe
top_3_productos = (
    validos
    .groupby("producto")["importe"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

top_3_productos

producto
Sensor A    819.0
Sensor B    416.0
Gateway     360.0
Name: importe, dtype: float64

In [11]:
# Encontrar id de los clientes que han hecho más de una compra
compras_por_cliente = (
    validos
    .groupby("cliente_id")
    .size()
)

clientes_repetidos = compras_por_cliente[compras_por_cliente > 1]

clientes_repetidos

cliente_id
C001    3
C002    2
dtype: int64

In [12]:
# Exportación. Convierte el dataframe validos en un archivo csv
ruta_ventas_limpias = Path("../Datos/ventas_limpias.csv")

validos.to_csv(ruta_ventas_limpias, index=False)

ruta_ventas_limpias

WindowsPath('../Datos/ventas_limpias.csv')

In [13]:
# Crear el json

import json

calidad_datos = {
    "filas_totales": len(ventas),
    "filas_validas": len(validos),
    "filas_invalidas": len(errores),
    "importe_total": float(validos["importe"].sum())
}

ruta_calidad = Path("../Datos/calidad_datos.json")

with open(ruta_calidad, "w", encoding="utf-8") as archivo:
    json.dump(calidad_datos, archivo, indent=4)

calidad_datos

{'filas_totales': 10,
 'filas_validas': 8,
 'filas_invalidas': 2,
 'importe_total': 1595.0}